# HumanAI Detect - Uretici-Dengeli Egitim Deneyi (Colab)

**Amac:** Havuzdaki uretici payi cok dengesiz (Qwen ~%89, GPT-4o-mini ~%17, Claude ~%3.5 -- ai_raw+ai_humanized toplaminda). Hipotez: modelin Claude'a zayif genellemesinin (n=15/15 testte %6.7) buyuk kismi bu dengesizlikten kaynaklaniyor. Bu notebook, Qwen'i ai_raw/ai_humanized icinde uretici basina ~1000 ornege (GPT-4o-mini'nin dogal payina yakin) alt-orneklenmis bir havuzla AYNI LOGO-CV'yi tekrarlar ve baseline (`colab_logo_cv.ipynb` ciktisi) ile yan yana karsilastirir. **Kucuk ureticiler (Claude ~209/418) ASLA yukari-orneklenip tekrarlanmiyor** -- sadece Qwen'in asiri fazlaligi kirpiliyor.

**ONKOSUL:** Once `colab_logo_cv.ipynb` calistirilip `logo_cv.json` indirilip yerlestirilmis olmali (bu notebook onu baseline olarak yukleyecek).

**ONEMLI -- Runtime tipi: CPU + Yuksek RAM secilmeli, GPU DEGIL.** Bu notebook'taki egitim (stacking: XGBoost/CatBoost/MLP/LogReg) GPU'ya ozel yapilandirilmadi -- GPU runtime'i secmek sadece daha az vCPU/RAM ayrilmasina yol acip yavaslatir (bkz. proje notlari, `colab_measure_calibration.ipynb`'de de ayni sebeple CPU+Yuksek RAM kullaniliyor). Runtime > Change runtime type > Hardware accelerator: None, Runtime shape: High-RAM.

**Girdi:** `data/processed/fused.parquet`, `outputs/reports/cv_results/logo_cv.json` (baseline).
**Cikti:** `logo_cv_generator_balanced.json/.md`, `generator_balance_comparison.md`.

In [ ]:
# Repo klonla (ilk kez) veya guncelle (klasor zaten varsa git pull)
GITHUB_REPO = 'https://github.com/BurakCANKURT/humanai-detect.git'
import os
if os.path.exists('/content/humanai_detect'):
    %cd /content/humanai_detect
    !git pull origin master
else:
    !git clone {GITHUB_REPO} /content/humanai_detect
    %cd /content/humanai_detect
!git log --oneline -5


In [ ]:
# Bagimliliklari kur (birkac dakika surebilir)
!pip install -e '.[dev]' -q


In [ ]:
# fused.parquet + baseline logo_cv.json yukle -- IKISINI BIRDEN sec
import os
os.makedirs('data/processed', exist_ok=True)
os.makedirs('outputs/reports/cv_results', exist_ok=True)
from google.colab import files
uploaded = files.upload()
import shutil
for fname in uploaded:
    if fname == 'fused.parquet':
        shutil.move(fname, f'data/processed/{fname}')
    elif fname == 'logo_cv.json':
        shutil.move(fname, f'outputs/reports/cv_results/{fname}')
    else:
        print(f'beklenmeyen dosya, elle tasi: {fname}')


In [ ]:
# Uretici-dengeli egitim + LOGO-CV + baseline karsilastirmasi (asil is)
!python scripts/train_generator_balanced.py --target-per-generator 1000


In [ ]:
# Karsilastirma tablosunu ekranda goster
print(open('outputs/reports/cv_results/generator_balance_comparison.md', encoding='utf-8').read())


In [ ]:
# Sonuclari yerel makineye indir
from google.colab import files
files.download('outputs/reports/cv_results/logo_cv_generator_balanced.json')
files.download('outputs/reports/cv_results/logo_cv_generator_balanced.md')
files.download('outputs/reports/cv_results/generator_balance_comparison.md')


## Indirilen Dosyalari Yerlestirme

Uc dosyayi da `outputs/reports/cv_results/` altina yerlestir, Claude'a haber ver. `generator_balance_comparison.md` dogrudan hipotezin cevabi: dengeleme Claude/GPT-4o-mini'nin katı dogrulugunu belirgin yukselttiyse (ozellikle Claude'da), dengesiz uretici payinin asil sebep oldugu dogrulanmis olur -- bu durumda production egitiminde de bu dengeleme kalici olarak benimsenebilir.